# 04 · Optimización de hiperparámetros
Optimizamos el mejor modelo supervisado (Regresión Logística) con **GridSearchCV** y **RandomizedSearchCV**, y medimos el impacto.

In [ ]:
# === Setup reproducible (Google Colab) ===

import sys, os, subprocess, glob
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

hits = glob.glob("/content/drive/MyDrive/**/src/config.py", recursive=True)
if not hits:
    raise FileNotFoundError(
        "No se encontro src/config.py en tu Drive. Verifica que subiste "
        "la carpeta completa del proyecto (con la carpeta src/ y sus .py)."
    )
root = Path(hits[0]).parent.parent
os.environ["PROJECT_ROOT"] = str(root)
sys.path.insert(0, str(root / "src"))

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "scikit-learn", "pandas", "numpy", "matplotlib", "seaborn"],
               check=False)

from config import set_global_seed, ensure_dirs, SEED
set_global_seed(SEED)   # fija todas las semillas
ensure_dirs()           # crea carpetas de salida
print(f"Entorno listo. Raiz del proyecto: {root} | SEED={SEED}")

Mounted at /content/drive
Entorno listo. Raiz del proyecto: /content/drive/MyDrive/Programación Ev2/Adolfo | SEED=42


In [ ]:
import pandas as pd, joblib
from sklearn.model_selection import train_test_split
from data_preprocessing import load_dataset, get_text_target
from config import MODELS_DIR, METRICS_DIR

df = load_dataset(); X, y = get_text_target(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)

## 1. Línea base (sin optimizar)

In [ ]:
from model_training import get_supervised_models
from sklearn.metrics import f1_score
base = get_supervised_models()['LogReg']
base.fit(X_train, y_train)
base_f1 = f1_score(y_test, base.predict(X_test), average='macro')
print(f'F1 base (test): {base_f1:.4f}')

F1 base (test): 0.9366


## 2. GridSearchCV — búsqueda exhaustiva
Explora C, tamaño de vocabulario y rango de n-gramas en una rejilla acotada.

In [ ]:
from hyperparameter_tuning import tune_logreg_grid, summarize_search
grid = tune_logreg_grid(X_train, y_train, cv=5)
summarize_search(grid)

{'best_score_cv_f1': 0.9424,
 'best_params': {'clf__C': 1.0,
  'tfidf__max_features': 3000,
  'tfidf__ngram_range': (1, 2)},
 'n_candidates': 12}

## 3. RandomizedSearchCV — muestreo eficiente
Muestrea C de una distribución log-uniforme; suele hallar buenas zonas con menos evaluaciones que la rejilla.

In [ ]:
from hyperparameter_tuning import tune_logreg_random
rnd = tune_logreg_random(X_train, y_train, n_iter=15, cv=5)
summarize_search(rnd)

{'best_score_cv_f1': 0.9417,
 'best_params': {'clf__C': np.float64(0.6338653441536259),
  'tfidf__max_features': 5000,
  'tfidf__ngram_range': (1, 2)},
 'n_candidates': 15}

## 4. Impacto de la optimización

In [ ]:
best = grid.best_estimator_
opt_f1 = f1_score(y_test, best.predict(X_test), average='macro')
impacto = pd.DataFrame({
    'modelo': ['Base', 'GridSearchCV', 'RandomizedSearchCV'],
    'cv_f1': [None, round(grid.best_score_,4), round(rnd.best_score_,4)],
    'test_f1': [round(base_f1,4), round(opt_f1,4),
                round(f1_score(y_test, rnd.best_estimator_.predict(X_test), average='macro'),4)]
})
impacto.to_csv(METRICS_DIR / 'tuning_impact.csv', index=False)
impacto

,modelo,cv_f1,test_f1
0,Base,NaN,0.9366
1,GridSearchCV,0.9424,0.9370
2,RandomizedSearchCV,0.9417,0.9361


In [ ]:
joblib.dump(grid.best_estimator_, MODELS_DIR / 'best_supervised_tuned.joblib')
print('Modelo optimizado guardado.')

Modelo optimizado guardado.


**Lectura:** con un dataset tan separable el margen de mejora es pequeño, pero el ejercicio documenta el proceso correcto: validación cruzada, búsqueda reproducible (random_state) y comparación honesta base vs optimizado.